# Phase 1: Inference & Panel Diagnostics
## The Hook
To accurately measure the impact of promotions across the Dominick's Finer Foods chain, we first need to understand the underlying structure of our data. Retail demand is inherently noisy and highly heterogeneous. 
        
## The Setup
We load the raw transactional data and product metadata to analyze cross-sectional variance (Store-to-Store differences) and distribution skewness. This validates our choice of **Fixed Effects** and **Log Transformations** in the regression pipeline.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style configuration
sns.set_theme(style='whitegrid', context='talk')

import sys
sys.path.append('..')
from src.data.load_data import load_data
from src.data.preprocess import preprocess_data

# Load and Preprocess for Panel
transactions, products = load_data('../data/raw/wcer.csv', '../data/raw/upccer.csv')
df = preprocess_data(transactions, products)
df.head()


## 1. Distribution Skewness: Why We Used Log Transformations
Retail sales and prices are notoriously right-skewed. If we ran OLS on raw units, the residuals would violate homoscedasticity assumptions. Let's look at the density of Sales to justify the `log1p` approach.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw Sales Distribution
sns.histplot(df['Sales'].sample(100000), bins=50, ax=axes[0], color='#2E86AB')
axes[0].set_title('Raw Sales Distribution (Right-Skewed)')
axes[0].set_xlabel('Sales Volume')

# Log Sales Distribution
sns.histplot(np.log1p(df['Sales'].sample(100000)), bins=50, ax=axes[1], color='#E63946')
axes[1].set_title('Log(Sales) Distribution (Normalized)')
axes[1].set_xlabel('Log(Sales)')

plt.tight_layout()
plt.show()


## 2. Unobserved Heterogeneity: Why We Used Fixed Effects
A promotion at a high-volume suburban store will yield completely different absolute numbers than the exact same promotion at a low-volume urban store. If we do not control for this *Unobserved Heterogeneity*, our promotional coefficient will be deeply biased.


In [ ]:
# Sample top 15 stores by transaction volume for visualization
top_stores = df['Store_ID'].value_counts().nlargest(15).index
df_sample = df[df['Store_ID'].isin(top_stores)].copy()

plt.figure(figsize=(14, 7))
sns.boxplot(data=df_sample, x='Store_ID', y='Sales', order=top_stores, palette='viridis', showfliers=False)
plt.title('Baseline Sales Variance Across Top 15 Stores')
plt.xlabel('Store ID')
plt.ylabel('Weekly Sales')
plt.xticks(rotation=45)
plt.show()


### The Implication
The massive variance in baselines requires us to "absorb" the store indicator variables. This is why we used high-dimensional Fixed Effects (`AbsorbingLS`) to isolate the true marginal impact of `Promo` within each individual store's context.
